In [1]:
import numpy as np 
import pandas as pd 
import altair as alt
from vega_datasets import data
import json

In [2]:
season_2324 = pd.read_csv('/Users/kbai/Downloads/PL-season-2324.csv')
season_2425 = pd.read_csv('/Users/kbai/Downloads/PL-season-2425.csv')

In [3]:
season_2324.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,...,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR
0,11/08/23,Burnley,Man City,0,3,A,0,2,A,C Pawson,...,1,8,11,8,6,5,0,0,1,0
1,12/08/23,Arsenal,Nott'm Forest,2,1,H,2,0,H,M Oliver,...,7,2,12,12,8,3,2,2,0,0
2,12/08/23,Bournemouth,West Ham,1,1,D,0,0,D,P Bankes,...,5,3,9,14,10,4,1,4,0,0
3,12/08/23,Brighton,Luton,4,1,H,1,0,H,D Coote,...,12,3,11,12,6,7,2,2,0,0
4,12/08/23,Everton,Fulham,0,1,A,0,0,D,S Attwell,...,9,2,12,6,10,4,0,2,0,0


In [4]:
season_2324.columns

Index(['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG',
       'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HF', 'AF', 'HC', 'AC',
       'HY', 'AY', 'HR', 'AR'],
      dtype='object')

In [5]:
season_2425.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,...,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR
0,16/08/24,Man United,Fulham,1,0,H,0,0,D,R Jones,...,5,2,12,10,7,8,2,3,0,0
1,17/08/24,Ipswich,Liverpool,0,2,A,0,0,D,T Robinson,...,2,5,9,18,2,10,3,1,0,0
2,17/08/24,Arsenal,Wolves,2,0,H,1,0,H,J Gillett,...,6,3,17,14,8,2,2,2,0,0
3,17/08/24,Everton,Brighton,0,3,A,0,1,A,S Hooper,...,1,5,8,8,1,5,1,1,1,0
4,17/08/24,Newcastle,Southampton,1,0,H,1,0,H,C Pawson,...,1,4,15,16,3,12,2,4,1,0


## Question 1 

How does team performance differ between the
two seasons?

In [6]:
season_2324['Season'] = '2023-24'
season_2425['Season'] = '2024-25'

In [7]:
home_2324 = season_2324.copy()

home_2324 = home_2324.assign(
    Team=home_2324['HomeTeam'],
    GoalsScored=home_2324['FTHG'],
    GoalsConceded=home_2324['FTAG'],
    Win=(home_2324['FTR'] == 'H').astype(int),
    Draw=(home_2324['FTR'] == 'D').astype(int),
    Loss=(home_2324['FTR'] == 'A').astype(int),
    Points=(
        (home_2324['FTR'] == 'H') * 3 +
        (home_2324['FTR'] == 'D') * 1
    )
)

home_2324 = home_2324[['Team', 'Season', 'GoalsScored', 'GoalsConceded', 'Win', 'Draw', 'Loss', 'Points']]
home_2324.head()

,Team,Season,GoalsScored,GoalsConceded,Win,Draw,Loss,Points
0,Burnley,2023-24,0,3,0,0,1,0
1,Arsenal,2023-24,2,1,1,0,0,3
2,Bournemouth,2023-24,1,1,0,1,0,1
3,Brighton,2023-24,4,1,1,0,0,3
4,Everton,2023-24,0,1,0,0,1,0


In [9]:
away_2324 = season_2324.copy()

away_2324 = away_2324.assign(
    Team=away_2324['AwayTeam'],
    GoalsScored=away_2324['FTAG'],
    GoalsConceded=away_2324['FTHG'],
    Win=(away_2324['FTR'] == 'A').astype(int),
    Draw=(away_2324['FTR'] == 'D').astype(int),
    Loss=(away_2324['FTR'] == 'H').astype(int),
    Points=(
        (away_2324['FTR'] == 'A') * 3 +
        (away_2324['FTR'] == 'D') * 1
    )
)

away_2324 = away_2324[['Team', 'Season', 'GoalsScored', 'GoalsConceded', 'Win', 'Draw', 'Loss', 'Points']]
away_2324.head()

,Team,Season,GoalsScored,GoalsConceded,Win,Draw,Loss,Points
0,Man City,2023-24,3,0,1,0,0,3
1,Nott'm Forest,2023-24,1,2,0,0,1,0
2,West Ham,2023-24,1,1,0,1,0,1
3,Luton,2023-24,1,4,0,0,1,0
4,Fulham,2023-24,1,0,1,0,0,3


In [10]:
team_matches_2324 = pd.concat([home_2324, away_2324], ignore_index=True)

In [11]:
team_summary_2324 = (
    team_matches_2324
    .groupby(['Team', 'Season'], as_index=False)
    .agg({
        'Points': 'sum',
        'Win': 'sum',
        'GoalsScored': 'sum',
        'GoalsConceded': 'sum'
    })
)

team_summary_2324['GoalDiff'] = (
    team_summary_2324['GoalsScored'] -
    team_summary_2324['GoalsConceded']
)

team_summary_2324.head()

,Team,Season,Points,Win,GoalsScored,GoalsConceded,GoalDiff
0,Arsenal,2023-24,89,28,91,29,62
1,Aston Villa,2023-24,68,20,76,61,15
2,Bournemouth,2023-24,48,13,54,67,-13
3,Brentford,2023-24,39,10,56,65,-9
4,Brighton,2023-24,48,12,55,62,-7


In [12]:
home_2425 = season_2425.copy()

home_2425 = home_2425.assign(
    Team=home_2425['HomeTeam'],
    GoalsScored=home_2425['FTHG'],
    GoalsConceded=home_2425['FTAG'],
    Win=(home_2425['FTR'] == 'H').astype(int),
    Draw=(home_2425['FTR'] == 'D').astype(int),
    Loss=(home_2425['FTR'] == 'A').astype(int),
    Points=(
        (home_2425['FTR'] == 'H') * 3 +
        (home_2425['FTR'] == 'D') * 1
    )
)

home_2425 = home_2425[['Team', 'Season', 'GoalsScored', 'GoalsConceded', 'Win', 'Draw', 'Loss', 'Points']]
home_2425.head()

,Team,Season,GoalsScored,GoalsConceded,Win,Draw,Loss,Points
0,Man United,2024-25,1,0,1,0,0,3
1,Ipswich,2024-25,0,2,0,0,1,0
2,Arsenal,2024-25,2,0,1,0,0,3
3,Everton,2024-25,0,3,0,0,1,0
4,Newcastle,2024-25,1,0,1,0,0,3


In [14]:
away_2425 = season_2425.copy()

away_2425 = away_2425.assign(
    Team=away_2425['AwayTeam'],
    GoalsScored=away_2425['FTAG'],
    GoalsConceded=away_2425['FTHG'],
    Win=(away_2425['FTR'] == 'A').astype(int),
    Draw=(away_2425['FTR'] == 'D').astype(int),
    Loss=(away_2425['FTR'] == 'H').astype(int),
    Points=(
        (away_2425['FTR'] == 'A') * 3 +
        (away_2425['FTR'] == 'D') * 1
    )
)

away_2425 = away_2425[['Team', 'Season', 'GoalsScored', 'GoalsConceded', 'Win', 'Draw', 'Loss', 'Points']]
away_2425.head()

,Team,Season,GoalsScored,GoalsConceded,Win,Draw,Loss,Points
0,Fulham,2024-25,0,1,0,0,1,0
1,Liverpool,2024-25,2,0,1,0,0,3
2,Wolves,2024-25,0,2,0,0,1,0
3,Brighton,2024-25,3,0,1,0,0,3
4,Southampton,2024-25,0,1,0,0,1,0


In [15]:
team_matches_2425 = pd.concat([home_2425, away_2425], ignore_index=True)

In [16]:
team_summary_2425 = (
    team_matches_2425
    .groupby(['Team', 'Season'], as_index=False)
    .agg({
        'Points': 'sum',
        'Win': 'sum',
        'GoalsScored': 'sum',
        'GoalsConceded': 'sum'
    })
)

team_summary_2425['GoalDiff'] = (
    team_summary_2425['GoalsScored'] -
    team_summary_2425['GoalsConceded']
)

team_summary_2425.head()

,Team,Season,Points,Win,GoalsScored,GoalsConceded,GoalDiff
0,Arsenal,2024-25,74,20,69,34,35
1,Aston Villa,2024-25,66,19,58,51,7
2,Bournemouth,2024-25,56,15,58,46,12
3,Brentford,2024-25,56,16,66,57,9
4,Brighton,2024-25,61,16,66,59,7


In [17]:
team_summary = pd.concat(
    [team_summary_2324, team_summary_2425],
    ignore_index=True
)
team_summary.head()

,Team,Season,Points,Win,GoalsScored,GoalsConceded,GoalDiff
0,Arsenal,2023-24,89,28,91,29,62
1,Aston Villa,2023-24,68,20,76,61,15
2,Bournemouth,2023-24,48,13,54,67,-13
3,Brentford,2023-24,39,10,56,65,-9
4,Brighton,2023-24,48,12,55,62,-7


In [18]:
team_summary_long = team_summary.melt(
    id_vars=['Team', 'Season'],
    value_vars=['Points', 'Win', 'GoalDiff'],
    var_name='Metric',
    value_name='Value'
)
team_summary_long.head()

,Team,Season,Metric,Value
0,Arsenal,2023-24,Points,89
1,Aston Villa,2023-24,Points,68
2,Bournemouth,2023-24,Points,48
3,Brentford,2023-24,Points,39
4,Brighton,2023-24,Points,48


Creating an interactive dropdown parameter that lets the user choose which performance metric (points, wins, or goal difference) the visualization will display:

In [19]:
metric_param = alt.param(
    name='SelectedMetric',
    bind=alt.binding_select(
        options=['Points', 'Win', 'GoalDiff'],
        name='Metric: '
    ),
    value='Points'
)

In [20]:
base = (
    alt.Chart(team_summary_long)
    .add_params(metric_param)
    .transform_filter(
        alt.datum.Metric == metric_param
    )
)

In [21]:
team_select = alt.selection_point(
    fields=['Team'],
    on='click',
    empty='all'
)

In [22]:
lines = (
    base
    .mark_line()
    .encode(
        x=alt.X('Season:N', title='Season'),
        y=alt.Y('Value:Q', title='Performance'),
        detail='Team:N',
        color=alt.condition(
            team_select,
            alt.value('#1f77b4'),   
            alt.value('lightgray')  
        ),
        opacity=alt.condition(
            team_select,
            alt.value(1),
            alt.value(0.3)
        )
    )
    .add_params(team_select)
)

In [23]:
points = (
    base
    .mark_circle(size=60)
    .encode(
        x='Season:N',
        y='Value:Q',
        detail='Team:N',
        color=alt.condition(
            team_select,
            alt.value('#1f77b4'),
            alt.value('lightgray')
        ),
        opacity=alt.condition(
            team_select,
            alt.value(1),
            alt.value(0.3)
        ),
        tooltip=[
            'Team:N',
            'Season:N',
            'Metric:N',
            'Value:Q'
        ]
    )
)


In [24]:
chart = (lines + points).properties(
    width=300,
    height=500,
    title='Team Performance Comparison Across Seasons (2023–24 vs 2024–25)'
)

chart

alt.LayerChart(...)

## Question 2

How consistent is a team’s attacking performance over time within a season?

In [25]:
matches = pd.concat([season_2324, season_2425], ignore_index=True)

In [26]:
home_attack = matches.assign(
    Team=matches['HomeTeam'],
    Goals=matches['FTHG'],
    Shots=matches['HS'],
    ShotsOnTarget=matches['HST'],
    Corners=matches['HC']
)

home_attack = home_attack[[
    'Team', 'Season', 'Date',
    'Goals', 'Shots', 'ShotsOnTarget', 'Corners'
]]

In [27]:
away_attack = matches.assign(
    Team=matches['AwayTeam'],
    Goals=matches['FTAG'],
    Shots=matches['AS'],
    ShotsOnTarget=matches['AST'],
    Corners=matches['AC']
)

away_attack = away_attack[[
    'Team', 'Season', 'Date',
    'Goals', 'Shots', 'ShotsOnTarget', 'Corners'
]]

In [28]:
team_match_attack = pd.concat(
    [home_attack, away_attack],
    ignore_index=True
)

In [29]:
team_match_attack.head()

,Team,Season,Date,Goals,Shots,ShotsOnTarget,Corners
0,Burnley,2023-24,11/08/23,0,6,1,6
1,Arsenal,2023-24,12/08/23,2,15,7,8
2,Bournemouth,2023-24,12/08/23,1,14,5,10
3,Brighton,2023-24,12/08/23,4,27,12,6
4,Everton,2023-24,12/08/23,0,19,9,10


In [30]:
team_match_attack['Date'] = pd.to_datetime(
    team_match_attack['Date'],
    format="%d/%m/%y"
)

In [31]:
team_match_attack = team_match_attack.sort_values(
    ['Team', 'Season', 'Date']
)

In [32]:
team_match_attack['Matchweek'] = (
    team_match_attack
    .groupby(['Team', 'Season'])
    .cumcount() + 1
)

In [33]:
team_match_attack = team_match_attack.sort_values(
    ['Team', 'Season', 'Matchweek']
)

team_match_attack['Goals_roll3'] = (
    team_match_attack
    .groupby(['Team', 'Season'])['Goals']
    .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)

team_match_attack['Shots_roll3'] = (
    team_match_attack
    .groupby(['Team', 'Season'])['Shots']
    .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)

team_match_attack['ShotsOnTarget_roll3'] = (
    team_match_attack
    .groupby(['Team', 'Season'])['ShotsOnTarget']
    .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)


In [34]:
team_match_attack[team_match_attack['Team'] == 'Arsenal'].head(6)

,Team,Season,Date,Goals,Shots,ShotsOnTarget,Corners,Matchweek,Goals_roll3,Shots_roll3,ShotsOnTarget_roll3
1,Arsenal,2023-24,2023-08-12,2,15,7,8,1,2.000000,15.000000,7.000000
778,Arsenal,2023-24,2023-08-21,1,14,3,8,2,1.500000,14.500000,5.000000
21,Arsenal,2023-24,2023-08-26,2,19,11,8,3,1.666667,16.000000,7.000000
38,Arsenal,2023-24,2023-09-03,3,17,5,12,4,2.000000,16.666667,6.333333
807,Arsenal,2023-24,2023-09-17,1,13,4,11,5,2.000000,16.333333,6.666667
54,Arsenal,2023-24,2023-09-24,2,13,6,11,6,2.000000,14.333333,5.000000


In [35]:
team_attack_long = team_match_attack.melt(
    id_vars=['Team', 'Season', 'Matchweek'],
    value_vars=[
        'Goals_roll3',
        'Shots_roll3',
        'ShotsOnTarget_roll3'
    ],
    var_name='Metric',
    value_name='RollingValue'
)

team_attack_long['Metric'] = (
    team_attack_long['Metric']
    .str.replace('_roll3', '', regex=False)
)

In [36]:
team_attack_long.head()

,Team,Season,Matchweek,Metric,RollingValue
0,Arsenal,2023-24,1,Goals,2.000000
1,Arsenal,2023-24,2,Goals,1.500000
2,Arsenal,2023-24,3,Goals,1.666667
3,Arsenal,2023-24,4,Goals,2.000000
4,Arsenal,2023-24,5,Goals,2.000000


Creating an interactive line chart that shows a selected team’s 3-match rolling average attacking performance (goals, shots, or shots on target) over matchweeks within a season.

In [37]:
attack_metric_param = alt.param(
    name='AttackMetric',
    bind=alt.binding_select(
        options=['Goals', 'Shots', 'ShotsOnTarget'],
        name='Attacking Metric: '
    ),
    value='Goals'
)

In [38]:
q2_chart = (
    alt.Chart(team_attack_long)
    .add_params(attack_metric_param)
    .transform_filter(team_select)
    .transform_filter(alt.datum.Metric == attack_metric_param)
    .mark_line()
    .encode(
        x=alt.X('Matchweek:Q', title='Matchweek'),
        y=alt.Y('RollingValue:Q', title='3-Match Rolling Average'),
        color=alt.value('#1f77b4'),
        tooltip=[
            'Team:N',
            'Season:N',
            'Matchweek:Q',
            'Metric:N',
            alt.Tooltip('RollingValue:Q', format='.2f')
        ]
    )
    .properties(
        width=500,
        height=350
    )
    .facet(
        column=alt.Column('Season:N', title='Season')
    )
    .properties(
        title='Attacking Consistency Within Each Season (3-Match Rolling Average)'
    )
)

dashboard = chart & q2_chart
dashboard

alt.VConcatChart(...)

## Question 3

How does home advantage manifest across
teams and seasons?

In [39]:
home_q3_2324 = season_2324.copy()
home_q3_2324['Season'] = '2023-24'

In [40]:
home_perf_2324 = pd.DataFrame({
    'Team': home_q3_2324['HomeTeam'],
    'Season': home_q3_2324['Season'],
    'Venue': 'Home',
    'GoalsScored': home_q3_2324['FTHG'],
    'GoalsConceded': home_q3_2324['FTAG']
})

home_perf_2324['Points'] = np.where(
    home_q3_2324['FTR'] == 'H', 3,
    np.where(home_q3_2324['FTR'] == 'D', 1, 0)
)

In [41]:
away_perf_2324 = pd.DataFrame({
    'Team': home_q3_2324['AwayTeam'],
    'Season': home_q3_2324['Season'],
    'Venue': 'Away',
    'GoalsScored': home_q3_2324['FTAG'],
    'GoalsConceded': home_q3_2324['FTHG']
})

away_perf_2324['Points'] = np.where(
    home_q3_2324['FTR'] == 'A', 3,
    np.where(home_q3_2324['FTR'] == 'D', 1, 0)
)

In [42]:
match_home_away_2324 = pd.concat(
    [home_perf_2324, away_perf_2324],
    ignore_index=True
)

match_home_away_2324.head()

,Team,Season,Venue,GoalsScored,GoalsConceded,Points
0,Burnley,2023-24,Home,0,3,0
1,Arsenal,2023-24,Home,2,1,3
2,Bournemouth,2023-24,Home,1,1,1
3,Brighton,2023-24,Home,4,1,3
4,Everton,2023-24,Home,0,1,0


In [43]:
home_q3_2425 = pd.DataFrame({
    'Team': season_2425['HomeTeam'],
    'Season': '2024-25',
    'Venue': 'Home',
    'GoalsScored': season_2425['FTHG'],
    'GoalsConceded': season_2425['FTAG']
})

home_q3_2425['Points'] = np.where(
    season_2425['FTR'] == 'H', 3,
    np.where(season_2425['FTR'] == 'D', 1, 0)
)

In [44]:
away_q3_2425 = pd.DataFrame({
    'Team': season_2425['AwayTeam'],
    'Season': '2024-25',
    'Venue': 'Away',
    'GoalsScored': season_2425['FTAG'],
    'GoalsConceded': season_2425['FTHG']
})

away_q3_2425['Points'] = np.where(
    season_2425['FTR'] == 'A', 3,
    np.where(season_2425['FTR'] == 'D', 1, 0)
)

In [45]:
match_home_away_q3_2425 = pd.concat(
    [home_q3_2425, away_q3_2425],
    ignore_index=True
)

match_home_away_q3_2425.head()

,Team,Season,Venue,GoalsScored,GoalsConceded,Points
0,Man United,2024-25,Home,1,0,3
1,Ipswich,2024-25,Home,0,2,0
2,Arsenal,2024-25,Home,2,0,3
3,Everton,2024-25,Home,0,3,0
4,Newcastle,2024-25,Home,1,0,3


In [46]:
match_home_away_q3 = pd.concat(
    [match_home_away_2324, match_home_away_q3_2425],
    ignore_index=True
)

match_home_away_q3.head()

,Team,Season,Venue,GoalsScored,GoalsConceded,Points
0,Burnley,2023-24,Home,0,3,0
1,Arsenal,2023-24,Home,2,1,3
2,Bournemouth,2023-24,Home,1,1,1
3,Brighton,2023-24,Home,4,1,3
4,Everton,2023-24,Home,0,1,0


In [47]:
q3_summary = (
    match_home_away_q3
    .groupby(['Team', 'Season', 'Venue'], as_index=False)
    .agg({
        'Points': 'sum',
        'GoalsScored': 'sum',
        'GoalsConceded': 'sum'
    })
)

q3_summary['GoalDiff'] = (
    q3_summary['GoalsScored'] -
    q3_summary['GoalsConceded']
)

q3_summary.head()

,Team,Season,Venue,Points,GoalsScored,GoalsConceded,GoalDiff
0,Arsenal,2023-24,Away,42,43,13,30
1,Arsenal,2023-24,Home,47,48,16,32
2,Arsenal,2024-25,Away,35,34,17,17
3,Arsenal,2024-25,Home,39,35,17,18
4,Aston Villa,2023-24,Away,28,28,33,-5


In [48]:
q3_pivot = (
    q3_summary
    .pivot(
        index=['Team', 'Season'],
        columns='Venue',
        values=['Points', 'GoalDiff']
    )
    .reset_index()
)

q3_pivot.head()

Team   Season Points      GoalDiff     
Venue                         Away Home     Away Home
0          Arsenal  2023-24     42   47       30   32
1          Arsenal  2024-25     35   39       17   18
2      Aston Villa  2023-24     28   40       -5   20
3      Aston Villa  2024-25     26   40       -7   14
4      Bournemouth  2023-24     21   27      -12   -1

In [49]:
q3_pivot.columns = [
    '_'.join(col).strip('_') if isinstance(col, tuple) else col
    for col in q3_pivot.columns
]

q3_pivot.head()

,Team,Season,Points_Away,Points_Home,GoalDiff_Away,GoalDiff_Home
0,Arsenal,2023-24,42,47,30,32
1,Arsenal,2024-25,35,39,17,18
2,Aston Villa,2023-24,28,40,-5,20
3,Aston Villa,2024-25,26,40,-7,14
4,Bournemouth,2023-24,21,27,-12,-1


In [50]:
q3_pivot.head()

,Team,Season,Points_Away,Points_Home,GoalDiff_Away,GoalDiff_Home
0,Arsenal,2023-24,42,47,30,32
1,Arsenal,2024-25,35,39,17,18
2,Aston Villa,2023-24,28,40,-5,20
3,Aston Villa,2024-25,26,40,-7,14
4,Bournemouth,2023-24,21,27,-12,-1


Building a brushed scatter plot comparing each team’s home vs. away points and linking it to a dynamic bar chart showing home advantage (home minus away points), then vertically combining them with the Q1 and Q2 views into the final dashboard.

In [51]:
brush = alt.param(select='interval')

In [52]:
q3_scatter = (
    alt.Chart(q3_pivot)
    .add_params(brush)
    .mark_circle(size=80)
    .encode(
        x=alt.X('Points_Away:Q', title='Away Points'),
        y=alt.Y('Points_Home:Q', title='Home Points'),
        color=alt.condition(
            brush,
            alt.value('#1f77b4'),
            alt.value('lightgray')
        ),
        tooltip=[
            'Team:N',
            'Season:N',
            'Points_Home:Q',
            'Points_Away:Q',
            'GoalDiff_Home:Q',
            'GoalDiff_Away:Q'
        ]
    )
    .properties(
        width=500,
        height=500,
        title='Home vs Away Performance'
    )
)

In [53]:
max_points = q3_pivot[['Points_Home', 'Points_Away']].max().max()

line_df = pd.DataFrame({
    'x': [0, max_points],
    'y': [0, max_points]
})

diag_line = (
    alt.Chart(line_df)
    .mark_line(color='black', strokeDash=[4,4])
    .encode(
        x='x:Q',
        y='y:Q'
    )
)

q3_scatter_with_line = alt.layer(
    q3_scatter,
    diag_line
)
q3_scatter_with_line

alt.LayerChart(...)

In [54]:
q3_pivot['HomeAdvantage'] = (
    q3_pivot['Points_Home'] - q3_pivot['Points_Away']
)

q3_pivot.head()

,Team,Season,Points_Away,Points_Home,GoalDiff_Away,GoalDiff_Home,HomeAdvantage
0,Arsenal,2023-24,42,47,30,32,5
1,Arsenal,2024-25,35,39,17,18,4
2,Aston Villa,2023-24,28,40,-5,20,12
3,Aston Villa,2024-25,26,40,-7,14,14
4,Bournemouth,2023-24,21,27,-12,-1,6


In [55]:
q3_bar = (
    alt.Chart(q3_pivot)
    .transform_filter(brush)
    .mark_bar()
    .encode(
        x=alt.X('Team:N', sort='-y'),
        y=alt.Y('HomeAdvantage:Q', title='Home Advantage (Points)'),
        color=alt.value('#1f77b4'),
        tooltip=['Team:N', 'Season:N', 'HomeAdvantage:Q']
    )
    .properties(
        width=500,
        height=300,
        title='Home Advantage (Brushed Teams)'
    )
)

In [56]:
q3_combined = alt.vconcat(
    q3_scatter_with_line,
    q3_bar
)
q3_combined

alt.VConcatChart(...)

## Final Dashboard:

**Analytical Questions Answered:**

1. How does team performance differ between the
two seasons?

2. How consistent is a team’s attacking performance over time within a season?

3. How does home advantage manifest across
teams and seasons?

In [57]:
final_dashboard = alt.vconcat(
    chart,       
    q2_chart,    
    q3_combined 
).resolve_scale(
    color='independent'
)

final_dashboard

alt.VConcatChart(...)

## Written Reflection

We want to understand your design process better.
Write a brief response on:

• How interactivity boosts perception of the visualizations you are presenting or amplifies the specific inferences you want the user to gather from
your design.

• What type of analyses would be lost from your design without such interactive provisions?

• What decisions did you have to make that weren’t outlined in the technical
specifications or suggestions?

• Why did you make those design choices?

• What if anything did you struggle with about this assignment?

Interactivity plays a key role in amplifying the inferences I want users to make across all three of the analytical questions I explore in the dashboard. In Question 1, where we explored how team performance differs between the two seasons, the metric dropdown allows users to dynamically switch between points, wins, and goal difference, making it easier to compare team performance across seasons without cluttering the view. Further, in Question 2, cross-filtering enables users to select a team in the season comparison view and immediately observe that team's attacking consistency over time, reinforcing connections between overall team performance and matchweek trends. There is also an additional dropdown menu that allows users to toggle the attacking metric between goals, shots, and shots on target, allowing users to do a deeper exploration of how different forms of attack evolve across matchweeks. In Question 3, where we explored how home advantage manifests across teams and seasons, the interval brush on the scatterplot of home points vs away points allows users to select subsets of teams based on their relative home versus away performance and see how that selection updates the bar chart of home advantage. Altogether, these interactions allow users to isolate patterns and uncover relationships across multiple levels of analysis rather than with a static presentation. 

Without these interactive provisions, much of the important analyses would be lost. For example, a static scatterplot and line chart would require the user to manually cross-reference team names and mentally filter subsets, which significantly increases cognitive load and reduces clarity. Further, removing the metric dropdown (points, wins, goal difference) from the scatterplot would significantly reduce the analytical depth of the visualization. A static chart showing only one metric would force users to view performance through a single lens, hiding meaningful differences in how teams succeed (if a team ranks highly in points but appear less strong in goal difference, or vice versa). Removing the interval brush specifically from Question 3's visualization would eliminate the ability for users to isolate clusters (such as teams with strong home records but weaker away results) and immediately see how that selection reshapes the bar chart of home advantage. This coordinated highlighting is what makes relative performance differences visually salient. The dashboard structure also enables comparisons across questions, allowing users to make connections between overall performance, seasonal patterns, and home advantage that would otherwise be far less intuitive in a purely static presentations. 

There were several design decisions not explicitly outlined in the technical instructions. I chose to include a diagonal reference line in the scatterplot to clearly mark parity between home and away points. I also decided to compute and visualize Home Advantage explicitly as a derived variable (difference between home and away points) rather than leaving users to infer it from raw point differences. While these design choices are relatively minimal, I made them mainly to reduce cognitive overload and guide interpretation. The diagonal line provides an immediate perceptual "benchmark" and computing Home Advantage directly supports clearer comparisons in the bar chart. 

The main challenge I encountered during this assignment was mainly implementing and debugging the brush interaction within the full dashboard, as well as structuring the dashboard so that multiple interactive components could coexist without overwhelming the user. Since each question included its own dropdowns or selection tools, I had to make sure that the controls, when displayed together, felt intuitive rather than cluttered. I also had to be intentional about the layout, spacing, and visual hierarchy of the dashboard because of this. Lastly, deciding how much information to display at once (especially when combining seasonsal comparisons, attacking metrics, and home advantage into a single dashboard) required balancing depth of analysis with visual clarity. Therefore, making sure that the dashboard was informative without becoming overwhelming was another design challenge. 